In [0]:

import pandas as pd
import plotly.express as px
from pyspark.sql import DataFrame, Window as W, functions as F, types as T
import json

# Как запускать ноутбук

1. В верхних параметрах выберите, что именно нужно сделать:
   * `RUN_ON_TEST_USERS` — запуск на тестовых пользователях или на рабочей выборке.
   * `LEVEL_COHORT_FRACTION` — шаг группировки уровней в когорты.
   * `BUILD_MM_RAW` — пересобрать витрину для MM.
   * `BUILD_MYM_RAW` — пересобрать витрину для MyM.
2. Если нужно только посмотреть статус уже готовых таблиц, оставьте `BUILD_MM_RAW = False` и `BUILD_MYM_RAW = False`.
3. Если нужна пересборка только одного проекта, включите флаг только для него.
4. Ноутбук рассчитан на запуск сверху вниз: каждая секция сама печатает понятный статус, поэтому ячейки можно свернуть и ориентироваться по выводу.
5. В финале ноутбук покажет последнюю доступную партицию по каждой выходной таблице.

In [0]:

# Считываем все управляющие параметры из верхней панели ноутбука.
# Эти параметры определяют, какие данные пересобирать и на какой выборке запускать расчёт.
RUN_ON_TEST_USERS = dbutils.widgets.get("RUN_ON_TEST_USERS") == "True"
LEVEL_COHORT_FRACTION = float(dbutils.widgets.get("LEVEL_COHORT_FRACTION"))
BUILD_MM_RAW = dbutils.widgets.get("BUILD_MM_RAW") == "True"
BUILD_MYM_RAW = dbutils.widgets.get("BUILD_MYM_RAW") == "True"

# Общие служебные переменные вынесены в начало, чтобы их было легко найти и поменять.
USER = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
MM_PATH = f"dbfs:/Users/{USER}/progress_analysis_dashboard/"
MYM_PATH = f"dbfs:/Users/{USER}/progress_analysis_dashboard/mym/"

if RUN_ON_TEST_USERS:
    MM_PATH = MM_PATH + "test_users/"
    MYM_PATH = MYM_PATH + "test_users/"

# Выходные таблицы, которые далее либо пересобираются, либо просто проверяются.
RAW_OBJECTS_MM_TABLE = "game_data_prod.analytics_voki.raw_objects_mm"
RAW_OBJECTS_MYM_TABLE = "game_data_prod.analytics_voki.raw_objects_mm_test_users"

print("Параметры запуска:")
print(f"  RUN_ON_TEST_USERS = {RUN_ON_TEST_USERS}")
print(f"  LEVEL_COHORT_FRACTION = {LEVEL_COHORT_FRACTION}")
print(f"  BUILD_MM_RAW = {BUILD_MM_RAW}")
print(f"  BUILD_MYM_RAW = {BUILD_MYM_RAW}")
print(f"  MM_PATH = {MM_PATH}")
print(f"  MYM_PATH = {MYM_PATH}")
print(f"  RAW_OBJECTS_MM_TABLE = {RAW_OBJECTS_MM_TABLE}")
print(f"  RAW_OBJECTS_MYM_TABLE = {RAW_OBJECTS_MYM_TABLE}")

In [0]:

def write_partitioned(df, path, partition_col="partition_date"):
  """Перезаписать DataFrame в parquet с партиционированием по дате."""
  (
    df.repartition(partition_col)
      .write.mode("overwrite")
      .partitionBy(partition_col)
      .parquet(path)
  )

# Секция 1. Сборка таблицы для MM

Ниже идут ячейки только для проекта **MM**.

Порядок внутри секции такой:
* сначала читаем источники;
* затем задаём константы и правила фильтрации;
* потом при необходимости пересобираем сырую витрину;
* если пересборка отключена, просто показываем статус уже существующей таблицы.

In [0]:
# Шаг 1. Подключаем входные таблицы для MM.
# Здесь только чтение источников: никаких тяжёлых расчётов и записей пока не происходит.
print("MM: подключаем источники данных")
print("MM working directory:", MM_PATH)

levels_mm = spark.table("bronze.levels_mm_amp")
user_state_mm = spark.table("silver.player_state_mm")
revenue_mm = spark.table("bronze.revenue_mm_amp")  # пока не используется, но оставлен для downstream

print("MM: источники успешно прочитаны")
print("  bronze.levels_mm_amp")
print("  silver.player_state_mm")
print("  bronze.revenue_mm_amp")
print(f"  выходная таблица: {RAW_OBJECTS_MM_TABLE}")

In [0]:
# Шаг 2. Задаём правила подготовки MM-данных.
# Эта ячейка описывает бизнес-логику: какие поля считаем ключами,
# как называем исходы матча и какую нижнюю границу по времени используем.

COLOR_MAP = {
    "close_fail": "#1f77b4",
    "far_fail":   "#d62728",
    "close_win":  "#ffcc00",
    "far_win":    "#2ca02c",
    "unknown":    "#7f7f7f",
}

OUTCOME_FLAGS_MM = {
    "FW": "far_win",
    "CW": "close_win",
    "CF": "close_fail",
    "FF": "far_fail",
}

WIN_PAIR = ["FW", "CW"]
FAIL_PAIR = ["FF", "CF"]

KEY_USER = ["client_time", "balance_id", "user_id", "partition_date"]
KEY_MAP = ["partition_date", "level_cohort"]
DATE_HARD_CUTOFF = "2026-01-01"

TEST_USERS_MM = [
    "17042026-082657-YpDIa7ZW", "18042026-082019-s5iwH0sC", "18032026-085553-ouCsFujZ",
    "17032026-024147-cRN1sFCc", "19032026-012513-ZRvSwYvq", "19032026-015621-0PNwoirg",
    "19032026-132942-XYcsux6B", "21032026-103441-GgG2vP4L", "22032026-131537-80SeC6aq",
    "22032026-140442-1vyKygOp",
]

print("MM: константы подготовлены")
print(f"  LEVEL_COHORT_FRACTION = {LEVEL_COHORT_FRACTION}")
print(f"  DATE_HARD_CUTOFF = {DATE_HARD_CUTOFF}")
print(f"  тестовых пользователей = {len(TEST_USERS_MM)}")

In [0]:
# # Шаг 3. Дополнительная справочная проверка по MM: какие AB-группы вообще есть в источнике.
# # Эта ячейка не влияет на пересборку витрины, она нужна только как быстрый ориентир по входным данным.

# ab_mm_rows = levels_mm.select("ab_group").distinct().collect()
# unique_ab_groups_mm = set()

# for row in ab_mm_rows:
#   ab_group = row.ab_group
#   if ab_group:
#     ab_group = json.loads(ab_group)
#     for group_name in ab_group:
#       unique_ab_groups_mm.add(group_name)

# unique_ab_groups_mm = sorted(unique_ab_groups_mm)
# unique_ab_groups_mm

### Что проверять после пересборки MM

После пересборки обычно смотрим:
* есть ли данные по дням без подозрительных провалов;
* корректно ли распределяются уровни по `level_cohort`;
* не пропали ли ожидаемые признаки и исходы матчей;
* если нужна отдельная аналитика по уровням с АП, её лучше добавлять ниже в черновые ячейки, не смешивая с основным пайплайном.

In [0]:
# Шаг 4. Основная логика MM.
if BUILD_MM_RAW:
  print("MM: запускаем пересборку raw-витрины")

  if RUN_ON_TEST_USERS:
    user_filter_mm = F.col("user_id").isin(TEST_USERS_MM)
  else:
    user_filter_mm = F.col("cheater").isin("Fair", "Soft")

  relevant_users_mm = (
    user_state_mm
    .filter(user_filter_mm)
    .select("user_id", "traffic_type")
    .distinct()
  )

  objects_mm = (
    levels_mm
    .filter(F.col("event_type") == "m3.level_finished")
    .filter(F.col("reason").isin("completed", "failed"))
    .filter(F.col("chain").isNull())
    .filter(F.col("client_time") > DATE_HARD_CUTOFF)
    .filter(F.col("partition_date") > DATE_HARD_CUTOFF)
    .join(F.broadcast(relevant_users_mm), on="user_id", how="inner")
    .withColumns({
      "failed": F.when(F.col("reason") == "failed", 1).otherwise(0),
      "balance_id": F.expr("try_cast(regexp_extract(get_json_object(event_payload, '$.balance_id'), 'st0*([0-9]+)', 1) AS INT)"),
    })
    .select(
      *KEY_USER,
      "traffic_type",
      "payer_type",
      "failed",
      F.json_tuple("event_payload", "reason_seg", "attempt", "super_ball").alias("reason_seg", "attempt", "super_ball"),
      F.json_tuple("user_payload", "tech.platform_name").alias("platform_name"),
    )
    .fillna("unknown", subset="reason_seg")
    .withColumns({
      "first_attempt": F.when(F.col("attempt") == 1, 1).otherwise(0),
      "level_cohort": F.floor(F.col("balance_id") / F.lit(LEVEL_COHORT_FRACTION)),
      "FW": F.when(F.col("reason_seg") == OUTCOME_FLAGS_MM["FW"], 1).otherwise(0),
      "CW": F.when(F.col("reason_seg") == OUTCOME_FLAGS_MM["CW"], 1).otherwise(0),
      "CF": F.when(F.col("reason_seg") == OUTCOME_FLAGS_MM["CF"], 1).otherwise(0),
      "FF": F.when(F.col("reason_seg") == OUTCOME_FLAGS_MM["FF"], 1).otherwise(0),
    })
  )

  (
    objects_mm.write
    .mode("overwrite")
    .option("mergeSchema", "true")
    .partitionBy("partition_date")
    .saveAsTable(RAW_OBJECTS_MM_TABLE)
  )

  print(f"MM: таблица успешно пересобрана -> {RAW_OBJECTS_MM_TABLE}")
  display(
    spark.table(RAW_OBJECTS_MM_TABLE)
      .agg(F.max("partition_date").alias("latest_partition"))
  )

  if RUN_ON_TEST_USERS:
    df_mm = spark.table(RAW_OBJECTS_MM_TABLE).toPandas()
    display(df_mm)
    print(f"MM: размер тестового датасета = {len(df_mm)}")
    for user_id in TEST_USERS_MM:
      display(px.scatter(df_mm[df_mm.user_id == user_id], x="client_time", y="level_cohort", color="reason_seg"))
else:
  print(f"MM: пересборка пропущена, используем готовую таблицу {RAW_OBJECTS_MM_TABLE}")
  try:
    display(
      spark.table(RAW_OBJECTS_MM_TABLE)
        .agg(F.max("partition_date").alias("latest_partition"))
    )
  except Exception as error:
    print("MM: не удалось прочитать готовую таблицу. Если это первый запуск, включите BUILD_MM_RAW = True.")
    print(error)

# Секция 2. Сборка таблицы для MyM

Логика ниже повторяет ту же последовательность, что и в секции MM, но для проекта **MyM**.

Ключевое отличие: исходы матчей здесь приезжают из отдельной таблицы и присоединяются джойном.

In [0]:
# Шаг 1. Подключаем входные таблицы для MyM.
# Эта секция независима от MM и использует свой набор источников.
print("MyM: подключаем источники данных")
print("MyM working directory:", MYM_PATH)

from functools import reduce

levels_mym = spark.table("bronze.levels_mym_amp")
user_state_mym = spark.table("silver.player_state_mym")
revenue_mym = spark.table("bronze.revenue_mym_amp")  # пока не используется, но оставлен для downstream

# В MyM исходы матчей лежат в отдельной таблице, а не в event_payload.
OUTCOMES_TABLE = "game_data_prod.analytics_voki.dk_mym_outcomes_ML_churn_2026"

print("MyM: источники успешно прочитаны")
print("  bronze.levels_mym_amp")
print("  silver.player_state_mym")
print("  bronze.revenue_mym_amp")
print(f"  outcomes table: {OUTCOMES_TABLE}")
print(f"  выходная таблица: {RAW_OBJECTS_MYM_TABLE}")

In [0]:
# Шаг 2. Задаём правила подготовки MyM-данных.
# Основная бизнес-логика такая же, как в MM, но дополнительно нужен фильтр по дрейфу времени
# и отдельный список колонок с исходами из таблицы outcomes.

COLOR_MAP = {
    "close_fail": "#1f77b4",
    "far_fail":   "#d62728",
    "close_win":  "#ffcc00",
    "far_win":    "#2ca02c",
    "unknown":    "#7f7f7f",
}

OUTCOME_FLAGS_MYM = ["FW", "CW", "CF", "FF"]
WIN_PAIR = ["FW", "CW"]
FAIL_PAIR = ["FF", "CF"]
KEY_USER = ["client_time", "balance_id", "user_id", "partition_date"]
KEY_MAP = ["partition_date", "level_cohort"]
MAX_FUTURE_DRIFT = 3 * 86400
DATE_HARD_CUTOFF = "2026-01-01"

TEST_USERS_MYM = [
    "12042026-195125-vxVRqdYZ", "16042026-121733-FZRhGlZB", "08042026-115122-22VS1Pg2",
    "29032026-212527-hBDzzfIP", "23092023-194259-tt60IN4o", "16042026-145502-NrExEToN",
    "20112023-154120-bFIT5fkI",
]

print("MyM: константы подготовлены")
print(f"  LEVEL_COHORT_FRACTION = {LEVEL_COHORT_FRACTION}")
print(f"  MAX_FUTURE_DRIFT = {MAX_FUTURE_DRIFT}")
print(f"  DATE_HARD_CUTOFF = {DATE_HARD_CUTOFF}")
print(f"  тестовых пользователей = {len(TEST_USERS_MYM)}")

In [0]:
# Шаг 3. Дополнительная справочная проверка по MyM: какие AB-группы присутствуют во входе.
# Как и в MM, это необязательная аналитическая ячейка, а не часть обязательной пересборки.
import json

ab_mym_rows = levels_mym.select("ab_group").distinct().collect()
unique_ab_groups_mym = set()

for row in ab_mym_rows:
  ab_group = row.ab_group
  if ab_group:
    ab_group = json.loads(ab_group)
    for group_name in ab_group:
      unique_ab_groups_mym.add(group_name)

unique_ab_groups_mym = sorted(unique_ab_groups_mym)
unique_ab_groups_mym

In [0]:
# Шаг 4. Основная логика MyM.
# Здесь тоже есть два режима: либо пересобираем таблицу, либо только проверяем текущую готовую витрину.
if BUILD_MYM_RAW:
  print("MyM: запускаем пересборку raw-витрины")

  if RUN_ON_TEST_USERS:
    user_filter_mym = F.col("user_id").isin(TEST_USERS_MYM)
  else:
    user_filter_mym = F.col("cheater").isin("Fair", "Soft")

  relevant_users_mym = (
    user_state_mym
    .filter(user_filter_mym)
    .select("user_id", "traffic_type")
    .distinct()
  )

  outcomes_mym = spark.table(OUTCOMES_TABLE)
  outcome_keys_mym = [c for c in KEY_USER if c in outcomes_mym.columns]
  outcomes_mym = outcomes_mym.select(*outcome_keys_mym, *OUTCOME_FLAGS_MYM)

  objects_mym = (
    levels_mym
    .withColumn("drift_sec", F.col("client_time").cast("long") - F.col("event_time").cast("long"))
    .filter(F.col("client_time").isNotNull() & F.col("event_time").isNotNull())
    .filter(F.col("drift_sec") <= F.lit(MAX_FUTURE_DRIFT))
    .drop("drift_sec")
    .filter(F.col("event_type") == "m3.level_finished")
    .filter(F.col("reason").isin("completed", "failed"))
    .filter(~F.col("chain").like("WL_%"))
    .filter(F.col("client_time") > DATE_HARD_CUTOFF)
    .filter(F.col("partition_date") > DATE_HARD_CUTOFF)
    .join(F.broadcast(relevant_users_mym), on="user_id", how="inner")
    .withColumns({
      "failed": F.when(F.col("reason") == "failed", 1).otherwise(0),
      "balance_id": F.expr("CAST(regexp_extract(get_json_object(event_payload, '$.balance_id'), 'st0*([0-9]+)', 1) AS INT)"),
    })
    .select(
      *KEY_USER,
      "traffic_type",
      "payer_type",
      "failed",
      "ab_group",
      F.col("attempt").cast("int").alias("attempt"),
      F.json_tuple("user_payload", "tech.platform_name").alias("platform_name"),
      F.json_tuple("event_payload", "super_ball").alias("super_ball"),
    )
    .withColumn("first_attempt", F.when(F.col("attempt") == 1, 1).otherwise(0))
  )

  objects_mym = (
    objects_mym
    .join(outcomes_mym, on=outcome_keys_mym, how="left")
    .fillna(0, subset=OUTCOME_FLAGS_MYM)
    .withColumns({
      "reason_seg": F.when(F.col("FW") == 1, "far_win")
                     .when(F.col("CW") == 1, "close_win")
                     .when(F.col("CF") == 1, "close_fail")
                     .when(F.col("FF") == 1, "far_fail")
                     .otherwise("unknown"),
      "level_cohort": F.floor(F.col("balance_id") / F.lit(LEVEL_COHORT_FRACTION)),
    })
  )

  (
    objects_mym.write
    .mode("overwrite")
    .option("mergeSchema", "false")
    .option("overwriteSchema", "true")
    .partitionBy("partition_date")
    .saveAsTable(RAW_OBJECTS_MYM_TABLE)
  )

  print(f"MyM: таблица успешно пересобрана -> {RAW_OBJECTS_MYM_TABLE}")
  display(
    spark.table(RAW_OBJECTS_MYM_TABLE)
      .agg(F.max("partition_date").alias("latest_partition"))
  )

  if RUN_ON_TEST_USERS:
    df_mym = spark.table(RAW_OBJECTS_MYM_TABLE).toPandas()
    display(df_mym)
    print(f"MyM: размер тестового датасета = {len(df_mym)}")
    for user_id in TEST_USERS_MYM:
      display(px.scatter(df_mym[df_mym.user_id == user_id], x="client_time", y="balance_id", color="reason_seg", color_discrete_map=COLOR_MAP))
else:
  print(f"MyM: пересборка пропущена, используем готовую таблицу {RAW_OBJECTS_MYM_TABLE}")
  try:
    display(
      spark.table(RAW_OBJECTS_MYM_TABLE)
        .agg(F.max("partition_date").alias("latest_partition"))
    )
  except Exception as error:
    print("MyM: не удалось прочитать готовую таблицу. Если это первый запуск, включите BUILD_MYM_RAW = True.")
    print(error)

In [0]:
# Финальная сводка по результату запуска.
# Эта ячейка полезна как короткий итог: по ней сразу видно, какие таблицы доступны
# и до какой даты в них лежат данные.
def get_latest_partition(table_name: str):
  try:
    return (
      spark.table(table_name)
      .agg(F.max("partition_date").alias("latest_partition"))
      .collect()[0]["latest_partition"]
    )
  except Exception as error:
    return f"недоступно: {type(error).__name__}"

status_df = pd.DataFrame([
  {
    "project": "MM",
    "table_name": RAW_OBJECTS_MM_TABLE,
    "rebuild_requested": BUILD_MM_RAW,
    "latest_partition": get_latest_partition(RAW_OBJECTS_MM_TABLE),
  },
  {
    "project": "MyM",
    "table_name": RAW_OBJECTS_MYM_TABLE,
    "rebuild_requested": BUILD_MYM_RAW,
    "latest_partition": get_latest_partition(RAW_OBJECTS_MYM_TABLE),
  },
])

display(status_df)

### Черновик для разовых проверок

Ниже можно добавлять временные ячейки для ручного анализа.

Базовый сценарий ноутбука уже полностью собран выше: обычно достаточно выбрать параметры вверху и запустить всё сверху вниз.